In [2]:
import pandas as pd
import matplotlib.pyplot as plt

In [3]:
df_lic = pd.read_parquet("data/data_licences/data_licences.parquet")
df_med = pd.read_csv("data/data_medailles/data_medailles_jo.csv")
df_or = pd.read_csv("data/data_medailles/data_or_jo.csv")
df_argent = pd.read_csv("data/data_medailles/data_argent_jo.csv")
df_bronze = pd.read_csv("data/data_medailles/data_bronze_jo.csv")

In [4]:
print(df_med.columns)
df_lic.head()

Index(['code_sport', 'sport', '2024_or', '2020_or', '2016_or', '2024_argent',
       '2020_argent', '2016_argent', '2024_bronze', '2020_bronze',
       '2016_bronze', 'total_medailles_2016', 'total_medailles_2020',
       'total_medailles_2024'],
      dtype='object')


,code_2024,code_annee_n,codes_2016_2024,federation,annee,sexe,age,tranche_age,grande_tranche_age,region,departement_long,licences_annuelles,code_sport,code_dep
0,101,101,101,Fédération Française d'Athlétisme,2016,F,5,b - de 5 à 9 ans,1 - Enfants (1-13),84 - Auvergne-Rhône-Alpes,01 - Ain,1,ATH,01
1,101,101,101,Fédération Française d'Athlétisme,2016,F,6,b - de 5 à 9 ans,1 - Enfants (1-13),84 - Auvergne-Rhône-Alpes,01 - Ain,13,ATH,01
2,101,101,101,Fédération Française d'Athlétisme,2016,F,7,b - de 5 à 9 ans,1 - Enfants (1-13),84 - Auvergne-Rhône-Alpes,01 - Ain,28,ATH,01
3,101,101,101,Fédération Française d'Athlétisme,2016,F,8,b - de 5 à 9 ans,1 - Enfants (1-13),84 - Auvergne-Rhône-Alpes,01 - Ain,49,ATH,01
4,101,101,101,Fédération Française d'Athlétisme,2016,F,9,b - de 5 à 9 ans,1 - Enfants (1-13),84 - Auvergne-Rhône-Alpes,01 - Ain,76,ATH,01


In [5]:
print(df_med.columns)
df_med.head()

Index(['code_sport', 'sport', '2024_or', '2020_or', '2016_or', '2024_argent',
       '2020_argent', '2016_argent', '2024_bronze', '2020_bronze',
       '2016_bronze', 'total_medailles_2016', 'total_medailles_2020',
       'total_medailles_2024'],
      dtype='object')


,code_sport,sport,2024_or,2020_or,2016_or,2024_argent,2020_argent,2016_argent,2024_bronze,2020_bronze,2016_bronze,total_medailles_2016,total_medailles_2020,total_medailles_2024
0,ATH,Athlétisme,0.0,0.0,0.0,1.0,1.0,3.0,0.0,0.0,3.0,6.0,1.0,1.0
1,AVI,Aviron,0.0,1.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,2.0,2.0,0.0
2,BAD,Badminton,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,BAK,Basket-ball,0.0,0.0,0.0,3.0,1.0,0.0,0.0,1.0,0.0,0.0,2.0,3.0
4,BOX,Boxe,0.0,0.0,2.0,2.0,0.0,2.0,1.0,0.0,2.0,6.0,0.0,3.0


In [6]:
# Copie de sécurité
df_lic_agg = df_lic.copy()

# Renommage pour clarté
df_lic_agg = df_lic_agg.rename(columns={
    'licences_annuelles': 'nb_licencies'
})

# AGRÉGATION : somme sur toutes les dimensions fines
df_lic_agg = (
    df_lic_agg
    .groupby(['code_sport', 'annee'], as_index=False)
    .agg({
        'nb_licencies': 'sum'
    })
)


In [7]:
df_lic_agg.head()

,code_sport,annee,nb_licencies
0,ATH,2016,301976
1,ATH,2017,307018
2,ATH,2018,314692
3,ATH,2019,316749
4,ATH,2020,305914


In [8]:
def jo_reference(annee):
    if annee < 2020:
        return 2016
    elif annee < 2024:
        return 2020
    else:
        return 2024

df_lic_agg['jo'] = df_lic_agg['annee'].apply(jo_reference)


In [9]:
df_med_long = pd.concat([
    df_med[['code_sport', 'sport',
            '2016_or', '2016_argent', '2016_bronze', 'total_medailles_2016']]
        .rename(columns=lambda x: x.replace('2016_', ''))
        .assign(jo=2016),

    df_med[['code_sport', 'sport',
            '2020_or', '2020_argent', '2020_bronze', 'total_medailles_2020']]
        .rename(columns=lambda x: x.replace('2020_', ''))
        .assign(jo=2020),

    df_med[['code_sport', 'sport',
            '2024_or', '2024_argent', '2024_bronze', 'total_medailles_2024']]
        .rename(columns=lambda x: x.replace('2024_', ''))
        .assign(jo=2024),
], ignore_index=True)


In [10]:
df_med_long.head()

,code_sport,sport,or,argent,bronze,total_medailles_2016,jo,total_medailles_2020,total_medailles_2024
0,ATH,Athlétisme,0.0,3.0,3.0,6.0,2016,NaN,NaN
1,AVI,Aviron,1.0,0.0,1.0,2.0,2016,NaN,NaN
2,BAD,Badminton,0.0,0.0,0.0,0.0,2016,NaN,NaN
3,BAK,Basket-ball,0.0,0.0,0.0,0.0,2016,NaN,NaN
4,BOX,Boxe,2.0,2.0,2.0,6.0,2016,NaN,NaN


In [11]:
import numpy as np
import pandas as pd

# =========================
# 0) Helpers
# =========================
def jo_reference(annee: int) -> int:
    # Hypothèse "licences de l'année N = saison à partir de sept N (post JO été N)"
    if annee < 2020:
        return 2016
    elif annee < 2024:
        return 2020
    else:
        return 2024


def safe_div(a, b):   # permet de gerer le cas où on a un nombre de licencié nul ce qui évite des métriques divergentes (dans ce cas on pose = 0)
    return np.where(b == 0, 0.0, a / b)


# =========================
# 1) Build medals long table
# =========================
def build_med_long(df_med: pd.DataFrame) -> pd.DataFrame:
    # 2016
    m2016 = df_med[['code_sport', 'sport', '2016_or', '2016_argent', '2016_bronze', 'total_medailles_2016']].copy()
    m2016 = m2016.rename(columns={
        '2016_or': 'or', '2016_argent': 'argent', '2016_bronze': 'bronze', 'total_medailles_2016': 'total_medailles'
    })
    m2016['jo'] = 2016

    # 2020
    m2020 = df_med[['code_sport', 'sport', '2020_or', '2020_argent', '2020_bronze', 'total_medailles_2020']].copy()
    m2020 = m2020.rename(columns={
        '2020_or': 'or', '2020_argent': 'argent', '2020_bronze': 'bronze', 'total_medailles_2020': 'total_medailles'
    })
    m2020['jo'] = 2020

    # 2024
    m2024 = df_med[['code_sport', 'sport', '2024_or', '2024_argent', '2024_bronze', 'total_medailles_2024']].copy()
    m2024 = m2024.rename(columns={
        '2024_or': 'or', '2024_argent': 'argent', '2024_bronze': 'bronze', 'total_medailles_2024': 'total_medailles'
    })
    m2024['jo'] = 2024

    med_long = pd.concat([m2016, m2020, m2024], ignore_index=True)
    return med_long


# =========================
# 2) Build sport-year features from micro licences
# =========================
def build_lic_features(df_lic: pd.DataFrame) -> pd.DataFrame:
    df = df_lic.copy()

    # --- sanity: ensure expected columns exist
    required = ['code_sport', 'annee', 'licences_annuelles']
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"Colonnes manquantes dans df_lic: {missing}")

    # Standardize types
    df['annee'] = df['annee'].astype(int)
    df['licences_annuelles'] = pd.to_numeric(df['licences_annuelles'], errors='coerce').fillna(0.0)

    # Optional columns: handle if absent
    if 'sexe' not in df.columns:
        df['sexe'] = 'U'
    if 'age' not in df.columns:
        df['age'] = np.nan
    if 'grande_tranche_age' not in df.columns:
        df['grande_tranche_age'] = 'Unknown'
    if 'code_dep' not in df.columns:
        df['code_dep'] = '00'

    # --- basic flags
    df['is_femme'] = (df['sexe'] == 'F').astype(int)
    df['is_homme'] = (df['sexe'] == 'H').astype(int)

    # age buckets (tu peux ajuster si besoin)
    df['age'] = pd.to_numeric(df['age'], errors='coerce')
    df['is_jeune_lt14'] = ((df['age'] < 14) & df['age'].notna()).astype(int)
    df['is_jeune_14_17'] = ((df['age'] >= 14) & (df['age'] <= 17)).astype(int)
    df['is_adulte_18_34'] = ((df['age'] >= 18) & (df['age'] <= 34)).astype(int)
    df['is_adulte_35_49'] = ((df['age'] >= 35) & (df['age'] <= 49)).astype(int)
    df['is_senior_50p'] = ((df['age'] >= 50)).astype(int)

    # --- aggregate totals per sport-year
    g = df.groupby(['code_sport', 'annee'], as_index=False)

    # Total licences
    base = g.agg(nb_licencies=('licences_annuelles', 'sum'))

    # Sex shares
    sex = g.apply(lambda x: pd.Series({
        'nb_femmes': (x['licences_annuelles'] * x['is_femme']).sum(),
        'nb_hommes': (x['licences_annuelles'] * x['is_homme']).sum()
    })).reset_index()

  

    # Age weighted stats + shares (robuste)
    def _age_block(x):
        w = x['licences_annuelles'].to_numpy(dtype=float)
        a = x['age'].to_numpy(dtype=float)

        mask = ~np.isnan(a)
        w2 = w[mask]
        a2 = a[mask]

        if w2.sum() == 0:
            age_mean = np.nan
            age_std = np.nan
        else:
            age_mean = float((a2 * w2).sum() / w2.sum())
            age2_mean = float(((a2 ** 2) * w2).sum() / w2.sum())
            var = max(0.0, age2_mean - age_mean ** 2)
            age_std = float(np.sqrt(var))

        nb_lt14  = float((x.loc[x['is_jeune_lt14'] == 1, 'licences_annuelles']).sum())
        nb_14_17 = float((x.loc[x['is_jeune_14_17'] == 1, 'licences_annuelles']).sum())
        nb_18_34 = float((x.loc[x['is_adulte_18_34'] == 1, 'licences_annuelles']).sum())
        nb_35_49 = float((x.loc[x['is_adulte_35_49'] == 1, 'licences_annuelles']).sum())
        nb_50p   = float((x.loc[x['is_senior_50p'] == 1, 'licences_annuelles']).sum())

        return pd.Series({
            'age_mean': age_mean,
            'age_std': age_std,
            'nb_lt14': nb_lt14,
            'nb_14_17': nb_14_17,
            'nb_18_34': nb_18_34,
            'nb_35_49': nb_35_49,
            'nb_50p': nb_50p
        })

    age_stats = df.groupby(['code_sport', 'annee']).apply(_age_block).reset_index()




    # --- Grande tranche age distribution (wide)
    # sum licences by (sport, year, grande_tranche_age) then pivot
    tranche = (
        df.groupby(['code_sport', 'annee', 'grande_tranche_age'], as_index=False)
          .agg(tranche_lic=('licences_annuelles', 'sum'))
    )
    tranche_wide = tranche.pivot_table(
        index=['code_sport', 'annee'],
        columns='grande_tranche_age',
        values='tranche_lic',
        aggfunc='sum',
        fill_value=0.0
    ).reset_index()

    # rename tranche columns
    tranche_cols = [c for c in tranche_wide.columns if c not in ['code_sport', 'annee']]
    tranche_wide = tranche_wide.rename(columns={c: f"tranche_{str(c).strip().lower().replace(' ', '_')}" for c in tranche_cols})

    # --- Geography: number of active departments + concentration (Herfindahl)
    dept_sum = (
        df.groupby(['code_sport', 'annee', 'code_dep'], as_index=False)
          .agg(dep_lic=('licences_annuelles', 'sum'))
    )
    dept_feat = dept_sum.groupby(['code_sport', 'annee']).apply(lambda x: pd.Series({
        'nb_departements_actifs': (x['dep_lic'] > 0).sum(),
        # Herfindahl: sum of squared shares across departments (higher = more concentrated)
        'herfindahl_dep': (safe_div(x['dep_lic'].values, x['dep_lic'].sum()) ** 2).sum() if x['dep_lic'].sum() > 0 else 0.0
    })).reset_index()

    # --- Merge all feature blocks
    out = base.merge(sex, on=['code_sport', 'annee'], how='left') \
              .merge(age_stats, on=['code_sport', 'annee'], how='left') \
              .merge(tranche_wide, on=['code_sport', 'annee'], how='left') \
              .merge(dept_feat, on=['code_sport', 'annee'], how='left')

    # Shares from counts
    out['part_femmes'] = safe_div(out['nb_femmes'], out['nb_licencies'])
    out['part_hommes'] = safe_div(out['nb_hommes'], out['nb_licencies'])

    out['part_lt14'] = safe_div(out['nb_lt14'], out['nb_licencies'])
    out['part_14_17'] = safe_div(out['nb_14_17'], out['nb_licencies'])
    out['part_18_34'] = safe_div(out['nb_18_34'], out['nb_licencies'])
    out['part_35_49'] = safe_div(out['nb_35_49'], out['nb_licencies'])
    out['part_50p'] = safe_div(out['nb_50p'], out['nb_licencies'])

    # Drop intermediate absolute counts if you want (tu peux garder aussi)
    # out = out.drop(columns=['nb_femmes','nb_hommes','nb_lt14','nb_14_17','nb_18_34','nb_35_49','nb_50p'])

    # --- Lags and dynamics (NO leakage: only past years)
    out = out.sort_values(['code_sport', 'annee']).reset_index(drop=True)

    for lag in [1, 2]:
        out[f'nb_licencies_lag{lag}'] = out.groupby('code_sport')['nb_licencies'].shift(lag)
        out[f'part_femmes_lag{lag}'] = out.groupby('code_sport')['part_femmes'].shift(lag)
        out[f'age_mean_lag{lag}'] = out.groupby('code_sport')['age_mean'].shift(lag)

    out['croissance_lag1'] = safe_div(out['nb_licencies'] - out['nb_licencies_lag1'], out['nb_licencies_lag1'])
    out['croissance_lag2'] = safe_div(out['nb_licencies_lag1'] - out['nb_licencies_lag2'], out['nb_licencies_lag2'])

    # rolling mean of last 2 years (t-1,t-2) to predict t
    out['nb_licencies_roll2'] = (
        out.groupby('code_sport')['nb_licencies']
           .shift(1)
           .rolling(2)
           .mean()
           .reset_index(level=0, drop=True)
    )

    return out


# =========================
# 3) Final dataset builder
# =========================
def build_model_dataset(df_lic: pd.DataFrame, df_med: pd.DataFrame) -> tuple[pd.DataFrame, list, list]:
    # features from licences micro
    lic_feat = build_lic_features(df_lic)

    # medals long
    med_long = build_med_long(df_med)

    # attach JO reference to each sport-year observation
    lic_feat['jo'] = lic_feat['annee'].apply(jo_reference)
    lic_feat['annees_depuis_jo'] = lic_feat['annee'] - lic_feat['jo']

    # merge medals (sport, jo)
    df_model = lic_feat.merge(med_long, on=['code_sport', 'jo'], how='left')

    # categorical
    df_model['sport'] = df_model['sport'].astype('category')

    # optional target transforms
    df_model['log_nb_licencies'] = np.log1p(df_model['nb_licencies'])

    # choose features (tu peux en enlever/rajouter)
    tranche_feature_cols = [c for c in df_model.columns if c.startswith('tranche_')]
    features = [
        'sport', 'annee', 'annees_depuis_jo',
        'or', 'argent', 'bronze', 'total_medailles',
        'part_femmes', 'age_mean', 'age_std',
        'part_lt14', 'part_14_17', 'part_18_34', 'part_35_49', 'part_50p',
        'nb_departements_actifs', 'herfindahl_dep',
        'nb_licencies_lag1', 'nb_licencies_lag2', 'croissance_lag1', 'croissance_lag2', 'nb_licencies_roll2',
        'part_femmes_lag1', 'age_mean_lag1'
    ] + tranche_feature_cols

    # keep only columns that exist (robust)
    features = [c for c in features if c in df_model.columns]

    cat_features = ['sport']
    return df_model, features, cat_features


# =========================
# 4) RUN
# =========================
df_model, features, cat_features = build_model_dataset(df_lic, df_med)

# Exemple: garder une fenêtre temporelle si tu veux (sinon garde tout)
# df_model = df_model[(df_model['annee'] >= 2016) & (df_model['annee'] <= 2024)].copy()

print(df_model.shape)
print("Nb features:", len(features))
print("Quelques features:", features[:15])

# Target recommandé:
target = 'nb_licencies'         # ou 'log_nb_licencies'


(306, 44)
Nb features: 29
Quelques features: ['sport', 'annee', 'annees_depuis_jo', 'or', 'argent', 'bronze', 'total_medailles', 'part_femmes', 'age_mean', 'age_std', 'part_lt14', 'part_14_17', 'part_18_34', 'part_35_49', 'part_50p']


In [12]:
df_model.head()

,code_sport,annee,nb_licencies,index,nb_femmes,nb_hommes,age_mean,age_std,nb_lt14,nb_14_17,...,croissance_lag2,nb_licencies_roll2,jo,annees_depuis_jo,sport,or,argent,bronze,total_medailles,log_nb_licencies
0,ATH,2016,301976,0,141983,159993,28.460622,19.342358,105365.0,36104.0,...,NaN,NaN,2016,0,Athlétisme,0.0,3.0,3.0,6.0,12.618106
1,ATH,2017,307018,1,146532,160486,28.641338,19.515100,106684.0,36941.0,...,NaN,NaN,2016,1,Athlétisme,0.0,3.0,3.0,6.0,12.634665
2,ATH,2018,314692,2,147945,165753,28.724186,19.756976,110883.0,36432.0,...,0.016697,304497.0,2016,2,Athlétisme,0.0,3.0,3.0,6.0,12.659353
3,ATH,2019,316749,3,150440,166309,29.010014,19.952825,111257.0,35764.0,...,0.024995,310855.0,2016,3,Athlétisme,0.0,3.0,3.0,6.0,12.665868
4,ATH,2020,305914,4,144038,161876,29.511235,20.206277,106195.0,33327.0,...,0.006537,315720.5,2020,0,Athlétisme,0.0,1.0,0.0,1.0,12.631063


In [ ]:
import numpy as np
import pandas as pd

from catboost import CatBoostRegressor, Pool
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# =========================
# 1) Choix target + features
# =========================
TARGET = 'nb_licencies'   # ou 'log_nb_licencies' si tu veux stabiliser la variance

# Si tu as déjà `features` et `cat_features`, garde-les.
# Sinon, sélection automatique (on évite les ids "trop identifiants")
if 'features' not in globals():
    drop_cols = {TARGET}
    drop_cols |= {'log_nb_licencies'} if TARGET != 'log_nb_licencies' and 'log_nb_licencies' in df_model.columns else set()

    # On retire les colonnes non-features évidentes si elles existent
    drop_cols |= {'nb_femmes','nb_hommes','nb_lt14','nb_14_17','nb_18_34','nb_35_49','nb_50p'}

    # Garde tout le reste
    features = [c for c in df_model.columns if c not in drop_cols]

# Cat features
if 'cat_features' not in globals():
    cat_features = []
    for c in ['sport', 'code_sport']:
        if c in df_model.columns:
            cat_features.append(c)

# Assure types cat
for c in cat_features:
    df_model[c] = df_model[c].astype('category')

# =========================
# 2) Split temporel (recommandé)
# =========================
df_model = df_model.sort_values(['annee', 'code_sport']).reset_index(drop=True)

train_df = df_model[df_model['annee'] <= 2023].copy()
test_df  = df_model[df_model['annee'] == 2024].copy()

# Option : si tu veux évaluer aussi 2020 comme test, adapte le split.
print("Train years:", train_df['annee'].min(), "->", train_df['annee'].max(), " | n=", len(train_df))
print("Test years:", test_df['annee'].min(), "->", test_df['annee'].max(), " | n=", len(test_df))

# =========================
# 3) Gestion des NaN (lags)
# =========================
# CatBoost supporte NaN, donc pas besoin d'imputer.
# Mais si tu veux éviter que les premières années "sans lag" perturbent trop :
# on peut imputer seulement les lags (optionnel).
lag_cols = [c for c in features if 'lag' in c or 'roll' in c]
for c in lag_cols:
    if c in train_df.columns:
        # imputation simple : 0 (ou median par sport, à toi de choisir)
        train_df[c] = train_df[c].fillna(0)
        test_df[c]  = test_df[c].fillna(0)

# =========================
# 4) Pools CatBoost
# =========================
X_train = train_df[features]
y_train = train_df[TARGET]

X_test  = test_df[features]
y_test  = test_df[TARGET]

# --- 0) Sécurise cat_features : uniquement des noms de colonnes valides
cat_features = [c for c in cat_features if isinstance(c, str) and c in train_df.columns]

# --- 1) Remplace NaN dans les colonnes catégorielles + cast en str (CatBoost n'accepte pas NaN en cat)
for c in cat_features:
    train_df[c] = train_df[c].astype('string').fillna('MISSING_CAT').astype(str)
    test_df[c]  = test_df[c].astype('string').fillna('MISSING_CAT').astype(str)

# --- 2) (recommandé) Remplir les NaN des colonnes médailles si besoin
med_cols = ['or', 'argent', 'bronze', 'total_medailles']
for c in med_cols:
    if c in train_df.columns:
        train_df[c] = train_df[c].fillna(0)
        test_df[c]  = test_df[c].fillna(0)

# --- 3) Reconstruire X/y après les modifications
X_train = train_df[features]
X_test  = test_df[features]

# Si tu entraînes en log :
y_train = train_df['log_nb_licencies']
y_test  = test_df['log_nb_licencies']



train_pool = Pool(X_train, y_train, cat_features=cat_features)
test_pool  = Pool(X_test, y_test, cat_features=cat_features)

# =========================
# 5) Entraînement CatBoost
# =========================
model = CatBoostRegressor(
    loss_function='RMSE',
    iterations=2000,
    learning_rate=0.03,
    depth=6,
    random_seed=42,
    eval_metric='RMSE',
    od_type='Iter',          # early stopping
    od_wait=100,
    verbose=200
)

model.fit(
    train_pool,
    eval_set=test_pool,
    use_best_model=True
)

# =========================
# 6) Évaluation
# =========================
pred_test = model.predict(X_test)



# =========================
# 7) Feature importance
# =========================
imp = model.get_feature_importance(train_pool)
imp_df = pd.DataFrame({'feature': features, 'importance': imp}).sort_values('importance', ascending=False)
print("\nTop 20 features:")
print(imp_df.head(20))

# (Optionnel) Sauvegarde modèle
# model.save_model("catboost_licences.cbm")


Train years: 2016 -> 2023  | n= 272
Test years: 2024 -> 2024  | n= 34
0:	learn: 1.5532297	test: 1.5586237	best: 1.5586237 (0)	total: 149ms	remaining: 4m 58s
200:	learn: 0.0803233	test: 0.1768234	best: 0.1768234 (200)	total: 5.63s	remaining: 50.4s
400:	learn: 0.0372945	test: 0.1457061	best: 0.1457061 (400)	total: 11.1s	remaining: 44.2s
600:	learn: 0.0215809	test: 0.1394882	best: 0.1394845 (598)	total: 17.2s	remaining: 40.1s
800:	learn: 0.0134556	test: 0.1371008	best: 0.1371008 (800)	total: 23.1s	remaining: 34.6s
1000:	learn: 0.0089965	test: 0.1362828	best: 0.1362275 (985)	total: 28.6s	remaining: 28.6s
1200:	learn: 0.0061700	test: 0.1359458	best: 0.1359458 (1200)	total: 34.2s	remaining: 22.7s
1400:	learn: 0.0043395	test: 0.1358243	best: 0.1358228 (1377)	total: 39.7s	remaining: 17s
1600:	learn: 0.0031471	test: 0.1357099	best: 0.1357041 (1595)	total: 46.4s	remaining: 11.6s
1800:	learn: 0.0022946	test: 0.1355937	best: 0.1355928 (1797)	total: 54.3s	remaining: 6s
1999:	learn: 0.0017457	test: 

In [14]:
######## 1: baseline: prédire 2024 avec 2023

import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Vrai en niveau
y_true = np.expm1(test_df['log_nb_licencies'].values)

# Baseline = valeur lag-1
y_pred_baseline = test_df['nb_licencies_lag1'].fillna(0).values

rmse_b = np.sqrt(mean_squared_error(y_true, y_pred_baseline))
mae_b  = mean_absolute_error(y_true, y_pred_baseline)
mape_b = np.mean(np.abs((y_true - y_pred_baseline) / np.maximum(y_true, 1))) * 100
r2_b   = r2_score(y_true, y_pred_baseline)

print("\n=== BASELINE (y = y(t-1)) ===")
print(f"RMSE : {rmse_b:,.0f}")
print(f"MAE  : {mae_b:,.0f}")
print(f"MAPE : {mape_b:.2f}%")
print(f"R²   : {r2_b:.4f}")



=== BASELINE (y = y(t-1)) ===
RMSE : 48,333
MAE  : 20,945
MAPE : 4.46%
R²   : 0.9985


In [15]:
####### 2: model sans médailles


from catboost import CatBoostRegressor, Pool

# Supprimer les médailles
medal_cols = ['or', 'argent', 'bronze', 'total_medailles']
features_no_medals = [c for c in features if c not in medal_cols]

X_train_nm = train_df[features_no_medals]
X_test_nm  = test_df[features_no_medals]

y_train_log = train_df['log_nb_licencies']
y_test_log  = test_df['log_nb_licencies']

train_pool_nm = Pool(X_train_nm, y_train_log, cat_features=cat_features)
test_pool_nm  = Pool(X_test_nm,  y_test_log,  cat_features=cat_features)

model_no_medals = CatBoostRegressor(
    loss_function='RMSE',
    iterations=2000,
    learning_rate=0.03,
    depth=6,
    random_seed=42,
    od_type='Iter',
    od_wait=100,
    verbose=False
)

model_no_medals.fit(train_pool_nm, eval_set=test_pool_nm, use_best_model=True)

# Prédictions
pred_log_nm = model_no_medals.predict(X_test_nm)
y_pred_nm = np.expm1(pred_log_nm)

rmse_nm = np.sqrt(mean_squared_error(y_true, y_pred_nm))
mae_nm  = mean_absolute_error(y_true, y_pred_nm)
mape_nm = np.mean(np.abs((y_true - y_pred_nm) / np.maximum(y_true, 1))) * 100
r2_nm   = r2_score(y_true, y_pred_nm)

print("\n=== CATBOOST SANS MÉDAILLES ===")
print(f"RMSE : {rmse_nm:,.0f}")
print(f"MAE  : {mae_nm:,.0f}")
print(f"MAPE : {mape_nm:.2f}%")
print(f"R²   : {r2_nm:.4f}")



=== CATBOOST SANS MÉDAILLES ===
RMSE : 121,218
MAE  : 40,288
MAPE : 7.52%
R²   : 0.9903


In [17]:
#  3. modele avec médailles
# 
# Prédictions déjà calculées avant
pred_log_full = model.predict(X_test)
y_pred_full = np.expm1(pred_log_full)

rmse_f = np.sqrt(mean_squared_error(y_true, y_pred_full))
mae_f  = mean_absolute_error(y_true, y_pred_full)
mape_f = np.mean(np.abs((y_true - y_pred_full) / np.maximum(y_true, 1))) * 100
r2_f   = r2_score(y_true, y_pred_full)

print("\n=== CATBOOST AVEC MÉDAILLES ===")
print(f"RMSE : {rmse_f:,.0f}")
print(f"MAE  : {mae_f:,.0f}")
print(f"MAPE : {mape_f:.2f}%")
print(f"R²   : {r2_f:.4f}")



=== CATBOOST AVEC MÉDAILLES ===
RMSE : 219,203
MAE  : 66,254
MAPE : 9.10%
R²   : 0.9683


In [18]:
#### comparaison finale

import pandas as pd

summary = pd.DataFrame({
    'Model': ['Baseline lag1', 'CatBoost sans médailles', 'CatBoost avec médailles'],
    'RMSE': [rmse_b, rmse_nm, rmse_f],
    'MAE':  [mae_b,  mae_nm,  mae_f],
    'MAPE (%)': [mape_b, mape_nm, mape_f],
    'R2':   [r2_b,   r2_nm,   r2_f]
})

print("\n=== COMPARAISON DES MODÈLES (2024) ===")
print(summary)



=== COMPARAISON DES MODÈLES (2024) ===
                     Model           RMSE           MAE  MAPE (%)        R2
0            Baseline lag1   48333.387338  20945.411765  4.459086  0.998459
1  CatBoost sans médailles  121218.129114  40287.683725  7.517374  0.990309
2  CatBoost avec médailles  219203.436931  66254.204013  9.104402  0.968308


On voit que le modele avec les médailles donne le pire résultat sur 2024 et que l'inertie du nombre de licenciés prédomine ( y(t)~y(t-1) ). On peut donc se dire que les médailles dues aux jo n'ont pas un effet immédiat sur le nombres de licenciés. Deux pistes d'amélioration: regarder l'effet des jo 2020 sur le nombre de licenciés des années qui suivent + regarder l'impact des médailles obtenues aux compétitions or JO. On oura potentiellement essayer un split autres que temporel mais qui prend aussi en compte le sport, à voir si cela est pertinent

ON va commencer par essayer d'estimer l'effet des médailles aux JO 2023 que l'évolution du nombre de licenciés entre 2021 et 2023
Il faudrait aussi regarder si cerrtains sports sont mieux prédits que d'autres

In [19]:
df_effect = df_model.copy()

# On se limite aux années post JO 2020
df_effect = df_effect[(df_effect['annee'] >= 2021) & (df_effect['annee'] <= 2023)].copy()

# Cible : croissance annuelle
df_effect['growth'] = (
    (df_effect['nb_licencies'] - df_effect['nb_licencies_lag1'])
    / df_effect['nb_licencies_lag1']
)

# Nettoyage
df_effect = df_effect.replace([np.inf, -np.inf], np.nan)
df_effect = df_effect.dropna(subset=['growth'])


In [22]:
# Liste "idéale" de features JO
candidate_features_jo = [
    'sport',
    'annees_depuis_jo',
    'or', 'argent', 'bronze', 'total_medailles',
    'part_femmes_lag1',
    'age_mean_lag1',
    'part_14_17_lag1',
    'nb_departements_actifs'
]

# On ne garde QUE celles qui existent vraiment
features_jo = [c for c in candidate_features_jo if c in df_effect.columns]

print("Features JO utilisées :")
print(features_jo)



Features JO utilisées :
['sport', 'annees_depuis_jo', 'or', 'argent', 'bronze', 'total_medailles', 'part_femmes_lag1', 'age_mean_lag1', 'nb_departements_actifs']


In [23]:
train_df = df_effect[df_effect['annee'] <= 2022].copy()
test_df  = df_effect[df_effect['annee'] == 2023].copy()

X_train = train_df[features_jo]
y_train = train_df['growth']

X_test  = test_df[features_jo]
y_test  = test_df['growth']


In [24]:
from catboost import Pool

cat_features = ['sport'] if 'sport' in features_jo else []

for c in cat_features:
    X_train[c] = X_train[c].astype(str).fillna('MISSING')
    X_test[c]  = X_test[c].astype(str).fillna('MISSING')

train_pool = Pool(X_train, y_train, cat_features=cat_features)
test_pool  = Pool(X_test, y_test, cat_features=cat_features)


C:\Users\mcmoi\AppData\Local\Temp\ipykernel_8448\1126003500.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X_train[c] = X_train[c].astype(str).fillna('MISSING')
C:\Users\mcmoi\AppData\Local\Temp\ipykernel_8448\1126003500.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X_test[c]  = X_test[c].astype(str).fillna('MISSING')


In [25]:

model_jo = CatBoostRegressor(
    loss_function='RMSE',
    iterations=1500,
    learning_rate=0.05,
    depth=5,
    random_seed=42,
    verbose=200
)

model_jo.fit(train_pool, eval_set=test_pool, use_best_model=True)

0:	learn: 0.2621667	test: 0.0929612	best: 0.0929612 (0)	total: 14.9ms	remaining: 22.4s
200:	learn: 0.0913974	test: 0.1521423	best: 0.0838983 (6)	total: 6.11s	remaining: 39.5s
400:	learn: 0.0532807	test: 0.1591149	best: 0.0838983 (6)	total: 13.7s	remaining: 37.6s
600:	learn: 0.0318285	test: 0.1583209	best: 0.0838983 (6)	total: 20s	remaining: 29.9s
800:	learn: 0.0226089	test: 0.1593325	best: 0.0838983 (6)	total: 26.4s	remaining: 23.1s
1000:	learn: 0.0156348	test: 0.1612841	best: 0.0838983 (6)	total: 35s	remaining: 17.4s
1200:	learn: 0.0112052	test: 0.1612742	best: 0.0838983 (6)	total: 40.7s	remaining: 10.1s
1400:	learn: 0.0070579	test: 0.1622560	best: 0.0838983 (6)	total: 46.1s	remaining: 3.25s
1499:	learn: 0.0060746	test: 0.1623355	best: 0.0838983 (6)	total: 48.9s	remaining: 0us

bestTest = 0.08389825491
bestIteration = 6

Shrink model to first 7 iterations.


In [26]:
from sklearn.metrics import mean_absolute_error, r2_score
import numpy as np

pred = model_jo.predict(X_test)

print("=== Effet JO 2020 sur la croissance (2023) ===")
print("MAE :", mean_absolute_error(y_test, pred))
print("R²  :", r2_score(y_test, pred))

# Comparaison médaillés vs non médaillés
test_df = test_df.copy()
test_df['pred_growth'] = pred

print("\nCroissance moyenne observée :")
print(test_df.groupby(test_df['total_medailles'] > 0)['growth'].mean())

print("\nCroissance moyenne prédite :")
print(test_df.groupby(test_df['total_medailles'] > 0)['pred_growth'].mean())


=== Effet JO 2020 sur la croissance (2023) ===
MAE : 0.06865645949857746
R²  : 0.009892071475378073

Croissance moyenne observée :
total_medailles
False    0.081365
True     0.079982
Name: growth, dtype: float64

Croissance moyenne prédite :
total_medailles
False    0.089511
True     0.078268
Name: pred_growth, dtype: float64


Ici, on voit que le nombre de licencié total n'est pas bien prédit par le modèle ce qui peut etre du à une variation en fonction des sports. on peut faire l'hypothèse donc que le nombre de licenciés global n'est pas ce qui est le plus influencé pat le jo ( donc ne met pas les gens aux sport) (mais ceci est ABSOLUMENT à vérifier avec les stats desc) et qu'elle modifie plutot la répartition entre les sport. En tout cas, je vais aller essayer une modélsation entre sport dans un autre fichier pour voir si les résultats sont plus concluants qu'ici